# Globe3D Demo: Image Colored & Split Globe

This notebook demonstrates how to color a globe using an equirectangular image, hollow it, and split it for 3D printing.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from globe3d import (
    GlobeModel,
    GeographicGrid,
    GridDisplacer,
    ImageColourer,
    calculate_displacement_scale
)

## 1. Generate Dummy Image

In [ ]:
# Create a simple gradient image for demonstration
width, height = 360, 180
image = np.zeros((height, width, 3), dtype=np.float32)

# Horizontal gradient (Hue-like)
for x in range(width):
    image[:, x, 0] = x / width      # Red channel
    image[:, x, 1] = 1 - x / width  # Green channel

# Vertical gradient
for y in range(height):
    image[y, :, 2] = y / height     # Blue channel

import os
os.makedirs('../outputs', exist_ok=True)
plt.imsave('../outputs/demo_texture.png', image)
plt.imshow(image)
plt.title("Generated Equirectangular Image")
plt.show()

## 2. Generate & Displace Sphere

In [ ]:
model_radius_mm = 40.0
topo_units = 'm'  # ETOPO data is in meters

model = GlobeModel(
    n_points=50000,
    radius=model_radius_mm,
    hollow=True,
    inner_ratio=0.8,
)

# Load Grid (using dummy if not found)
topogrid = "../inputs/ETOPO_2022_v1_60s_N90W180_surface.nc"
try:
    grid_data = GeographicGrid.from_netcdf(topogrid, lat_var='lat', lon_var='lon', data_var='z')
except FileNotFoundError:
    print("Grid file not found. Using dummy data.")
    lats = np.linspace(-90, 90, 180)
    lons = np.linspace(-180, 180, 360)
    grid = np.zeros((180, 360))
    grid[45, 90] = 5000
    grid_data = GeographicGrid(lats, lons, grid)

# Displace
scale = calculate_displacement_scale(model_radius_mm, vertical_exagg=50, grid_units=topo_units)
model.outer.displace(GridDisplacer(grid_data, show_progress=True), scale=scale)

## 3. Color from Image

In [ ]:
img_colourer = ImageColourer('../outputs/demo_texture.png', show_progress=True)
model.outer.colour(img_colourer)

## 4. Hollow & Split and Export

In [ ]:
model.export_hemispheres(
    '../outputs/globe_image_top.obj',
    '../outputs/globe_image_bottom.obj',
    engine='manifold',
)
print("Saved globe_image_top.obj and globe_image_bottom.obj")